# Unsloth Core — Model Evaluation

Evaluate a trained NPC model against a baseline or run a standalone quality check.


In [ ]:
# @title ⚙️ Evaluation Configuration
# @markdown Configure model paths, NPC key, judge model, and question count.

# --- NPC Selection ---
NPC_KEY = "history_guide"  # @param ["history_guide", "chef_assistant", "astronomy_guide", "fitness_coach"]
# @markdown The NPC key in snake_case (e.g. `history_guide`, `chef_assistant`).

# --- Model Paths ---
BASELINE_GGUF = ""  # @param {type:"string"}
# @markdown Optional path to a baseline GGUF for side-by-side comparison. Leave empty for standalone evaluation.

CANDIDATE_GGUF = ""  # @param {type:"string"}
# @markdown Path to the candidate GGUF model to evaluate (required).

# --- Evaluation Parameters ---
NUM_QUESTIONS = 5  # @param {type:"integer"}
# @markdown Number of evaluation questions to sample.

JUDGE_MODEL = "gpt-4o-mini"  # @param {type:"string"}
# @markdown Judge model for LLM-as-judge scoring. Use `gpt-4o-mini`, `gpt-4o`, or a local Ollama model.

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"  # @param {type:"string"}
# @markdown Base HuggingFace model ID. Used for reference only in the evaluation context.

# --- Advanced Flags ---
USE_JUDGE = True  # @param {type:"boolean"}
# @markdown Enable LLM judge for side-by-side scoring.

USE_HTML_REPORT = True  # @param {type:"boolean"}
# @markdown Generate an HTML report with Chart.js visualizations (comparison mode only).

print("Evaluation configuration loaded:")
print(f"  NPC_KEY        = {NPC_KEY}")
print(f"  BASELINE_GGUF  = {BASELINE_GGUF or "(none \u2014 standalone mode)"}")
print(f"  CANDIDATE_GGUF = {CANDIDATE_GGUF}")
print(f"  NUM_QUESTIONS  = {NUM_QUESTIONS}")
print(f"  JUDGE_MODEL    = {JUDGE_MODEL}")
print(f"  BASE_MODEL     = {BASE_MODEL}")
print(f"  USE_JUDGE      = {USE_JUDGE}")
print(f"  USE_HTML_REPORT = {USE_HTML_REPORT}")


In [ ]:
# @title 📦 Setup: Clone Repository & Install Dependencies
# @markdown Mounts Google Drive, clones the repo, installs llama-cpp-python and deepeval.

import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/athargamedev/Unsloth_Core.git"
DRIVE_REPO_DIR = "/content/drive/MyDrive/Unsloth_Core"
FALLBACK_REPO_DIR = "/content/Unsloth_Core"

# --- Detect Runtime ---
is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

if is_colab:
    print("Running in Google Colab.")
    repo_dir = DRIVE_REPO_DIR
    use_persistent = True
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            print("Google Drive mounted. Using persistent storage:", repo_dir)
        else:
            raise OSError("Google Drive mounted but MyDrive is not accessible.")
    except Exception as e:
        repo_dir = FALLBACK_REPO_DIR
        use_persistent = False
        print(f"Drive mount failed: {e}")
        print("Using ephemeral storage:", repo_dir)

    # Clone or pull the repository
    try:
        if not os.path.exists(repo_dir):
            os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
            subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
            print("Cloned repository.")
        else:
            git_dir = os.path.join(repo_dir, ".git")
            if os.path.exists(git_dir) and os.path.isdir(git_dir):
                orig = os.getcwd()
                os.chdir(repo_dir)
                subprocess.run(["git", "pull"], check=False)
                os.chdir(orig)
                print("Pulled latest changes.")
            else:
                import shutil
                shutil.rmtree(repo_dir)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
                print("Re-cloned repository (existing dir was not a git repo).")
    except OSError as e:
        print(f"Filesystem error during clone: {e}")
        if use_persistent:
            print("Falling back to ephemeral storage...")
            repo_dir = FALLBACK_REPO_DIR
            if not os.path.exists(repo_dir):
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)

    os.chdir(repo_dir)
    print("Changed to repo root:", os.getcwd())

    # Install dependencies
    print("\nInstalling llama-cpp-python (CUDA-enabled on Colab T4)...")
    subprocess.run(
        ["pip", "install", "-q", "llama-cpp-python"],
        check=True,
    )
    subprocess.run(
        ["pip", "install", "-q", "deepeval"],
        check=True,
    )
    print("All dependencies installed.")

else:
    print("=" * 60)
    print("NOT RUNNING IN GOOGLE COLAB.")
    print("Open this notebook at https://colab.research.google.com/")
    print("with a GPU runtime for model inference.")
    print("=" * 60)
    print("Continuing in local environment (limited support).")
    # Attempt to find local repo root
    curr = Path(os.getcwd()).resolve()
    repo_dir = None
    for parent in [curr] + list(curr.parents):
        if (parent / "ucore").exists() and (parent / "scripts").exists():
            repo_dir = parent
            break
    if repo_dir:
        print("Found local repo root:", repo_dir)
        os.chdir(str(repo_dir))
    else:
        print("Warning: Could not find local repo root.")

print("\nWorking directory:", os.getcwd())


In [ ]:
# @title 🤖 Configure Judge Credentials
# @markdown If using an OpenAI judge model, provide the API key. For local Ollama judges, no credentials needed.

import os

is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

judge_is_openai = JUDGE_MODEL.startswith("gpt-") or JUDGE_MODEL.startswith("o")

if USE_JUDGE and judge_is_openai:
    print(f"Using OpenAI judge model: {JUDGE_MODEL}")

    # Try Colab secrets first
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key and is_colab:
        try:
            from google.colab import userdata  # type: ignore
            api_key = userdata.get("OPENAI_API_KEY")
            if api_key:
                os.environ["OPENAI_API_KEY"] = api_key
                print("Loaded OPENAI_API_KEY from Colab secrets.")
        except Exception:
            pass

    # Interactive prompt as fallback
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        key = getpass("Enter OpenAI API Key: ")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            print("OpenAI API key set from user input.")
        else:
            print("WARNING: No API key provided. Judge will be disabled.")
            USE_JUDGE = False

elif USE_JUDGE:
    print(f"Using local/Ollama judge model: {JUDGE_MODEL}")
    print("NOTE: If running in Colab, Ollama is not available by default.")
    print("Consider switching JUDGE_MODEL to gpt-4o-mini for Colab use.")

else:
    print("Judge disabled. Evaluation will use heuristics only.")

print(f"USE_JUDGE = {USE_JUDGE}")


In [ ]:
# @title 🤖 Verify GPU & GGUF Files
# @markdown Checks GPU availability and validates that the GGUF file paths exist.

import os
import torch
from pathlib import Path

print("=" * 60)
print("GPU & FILE VALIDATION")
print("=" * 60)

# --- GPU Check ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\nGPU detected: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
else:
    print(f"\nWARNING: No GPU detected. CPU inference will be very slow.")

# --- llama-cpp-python Check ---
try:
    import llama_cpp
    print(f"llama-cpp-python version: {llama_cpp.__version__}")
except ImportError:
    print("llama-cpp-python not installed. Run the Setup cell first.")

# --- Verify Candidate GGUF ---
if not CANDIDATE_GGUF:
    print("\nERROR: CANDIDATE_GGUF is required. Set it in the config cell.")

candidate_path = Path(CANDIDATE_GGUF)
if CANDIDATE_GGUF and not candidate_path.exists():
    print(f"\nWARNING: Candidate GGUF not found: {CANDIDATE_GGUF}")
    # Search for exported GGUFs for this NPC
    exports_dir = Path(f"exports/{NPC_KEY}")
    if exports_dir.exists():
        gguf_files = sorted(exports_dir.glob("*.gguf"))
        if gguf_files:
            print(f"\nAvailable GGUFs in {exports_dir}/:")
            for f in gguf_files:
                size_mb = f.stat().st_size / 1024**2
                print(f"  {f.name} ({size_mb:.1f} MB)")
    else:
        print(f"  (exports directory not found: {exports_dir})")
    print("\nPlease update CANDIDATE_GGUF in the config cell with a valid path.")
elif CANDIDATE_GGUF:
    size_mb = candidate_path.stat().st_size / 1024**2
    print(f"\nCandidate GGUF: {CANDIDATE_GGUF} ({size_mb:.1f} MB)")

# --- Verify Baseline GGUF (optional) ---
if BASELINE_GGUF:
    baseline_path = Path(BASELINE_GGUF)
    if baseline_path.exists():
        size_mb = baseline_path.stat().st_size / 1024**2
        print(f"Baseline GGUF:   {BASELINE_GGUF} ({size_mb:.1f} MB)")
    else:
        print(f"WARNING: Baseline GGUF not found: {BASELINE_GGUF}")
        print("  Evaluation will fall back to standalone mode.")

# --- Verify Subject Spec ---
spec_path = f"subjects/{NPC_KEY}.json"
if os.path.exists(spec_path):
    print(f"Subject spec:    {spec_path}")
else:
    print(f"WARNING: Subject spec not found: {spec_path}")

print("\nValidation complete.")


In [ ]:
# @title 🧪 Run Model Evaluation
# @markdown Evaluates the candidate model. If BASELINE_GGUF is set, runs side-by-side comparison.
# @markdown Otherwise runs standalone evaluation with per-question metrics.

import os
import sys
import json
import time
from pathlib import Path

python_bin = sys.executable
spec_path = f"subjects/{NPC_KEY}.json"
output_dir = Path(f"eval/reports/{NPC_KEY}")
output_dir.mkdir(parents=True, exist_ok=True)
report_md = output_dir / "eval_report.md"
report_html = output_dir / "eval_report.html"
feedback_json_path = output_dir / "feedback.json"
standalone_json_path = output_dir / "standalone_results.json"

print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

if not CANDIDATE_GGUF:
    print("ERROR: CANDIDATE_GGUF is required. Set it in the config cell and re-run.")
    raise SystemExit(1)

if not os.path.exists(CANDIDATE_GGUF):
    print(f"ERROR: Candidate GGUF not found: {CANDIDATE_GGUF}")
    raise SystemExit(1)

# ────────────────────────────────────────
# Side-by-side comparison mode
# ────────────────────────────────────────
if BASELINE_GGUF:
    baseline_path = Path(BASELINE_GGUF)
    if not baseline_path.exists():
        print(f"ERROR: Baseline GGUF not found: {BASELINE_GGUF}")
        raise SystemExit(1)

    print(f"Mode: Side-by-side comparison")
    print(f"  Baseline:  {BASELINE_GGUF}")
    print(f"  Candidate: {CANDIDATE_GGUF}")
    print(f"  Questions: {NUM_QUESTIONS}")
    print(f"  Judge:     {JUDGE_MODEL if USE_JUDGE else '(disabled)'}")
    print()

    # Build command
    cmd = [
        python_bin, "scripts/evaluation/evaluate.py",
        "--baseline", str(baseline_path.resolve()),
        "--candidate", str(Path(CANDIDATE_GGUF).resolve()),
        "--spec", spec_path,
        "--num-questions", str(NUM_QUESTIONS),
        "--max-tokens", "256",
        "--port", str(9000),  # Avoid 8888 which is Jupyter's port
        "--output", str(report_md),
    ]
    if USE_JUDGE:
        cmd.extend(["--judge", "--judge-model", JUDGE_MODEL])
    if USE_HTML_REPORT:
        cmd.append("--report-html")
    cmd.extend(["--feedback-json", str(feedback_json_path)])

    print("Running evaluate.py...")
    print("Command:", " ".join(str(c) for c in cmd))
    print()

    result = subprocess.run(cmd, capture_output=True, text=True)

    # Always print stdout
    if result.stdout:
        print(result.stdout)

    # Print stderr if it contains useful info (not just server noise)
    if result.stderr:
        print("STDERR:")
        print(result.stderr[:3000])
        if len(result.stderr) > 3000:
            print(f"... (truncated, {len(result.stderr)} total chars)")

    if result.returncode != 0:
        print(f"\nERROR: Evaluation failed with exit code {result.returncode}")
        raise SystemExit(1)

    print("\nComparison evaluation complete.")

# ────────────────────────────────────────
# Standalone evaluation mode (no baseline)
# Uses evaluate.py's internal functions directly since the CLI
# requires both --baseline and --candidate for comparison mode.
# ────────────────────────────────────────
else:
    print(f"Mode: Standalone evaluation (no baseline)")
    print(f"  Model:     {CANDIDATE_GGUF}")
    print(f"  Questions: {NUM_QUESTIONS}")
    print()

    # Import evaluation functions directly
    sys.path.insert(0, os.getcwd())

    from scripts.evaluate import (
        LlamaServer,
        evaluate_model,
        load_subject_spec,
        generic_eval_questions,
        autodetect_validation_path,
        extract_questions_from_spec,
        check_sentence_count,
        check_no_ai_disclaimer,
        check_contains_name,
        identity_prompt_requires_name,
    )

    # Load spec and questions
    spec = None
    if os.path.exists(spec_path):
        spec = load_subject_spec(spec_path)
        val_path = autodetect_validation_path(spec.get("npc_key", NPC_KEY))
        questions = extract_questions_from_spec(spec, val_path)
        print(f"Loaded {len(questions)} questions from validation set.")
    else:
        questions = []

    if not questions:
        questions = generic_eval_questions(spec)
        print(f"Using {len(questions)} generic questions (no validation set found).")

    questions = questions[:NUM_QUESTIONS]
    print(f"Will evaluate {len(questions)} question(s).")
    print()

    # Start llama.cpp server
    print("Starting llama.cpp server...")
    server = LlamaServer(
        CANDIDATE_GGUF,
        port=9000,
        gpu_layers=99,
        max_tokens=256,
    )
    server_started = server.start()

    if not server_started:
        print("ERROR: Failed to start llama.cpp server.")
        print("Check that CANDIDATE_GGUF exists and is a valid GGUF file.")
        print("You can also try running with CPU offload by editing")
        print("the `gpu_layers` parameter in this cell.")
        raise SystemExit(1)

    # Run evaluation
    print()
    print("Evaluating model...")
    results = evaluate_model(server, questions, spec)
    server.stop()

    # Build structured output
    eval_data = {
        "npc_key": NPC_KEY,
        "model": CANDIDATE_GGUF,
        "num_questions": len(questions),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "results": [],
    }

    total_quality = 0
    total_words = 0
    total_latency = 0
    constraint_ok = 0

    for i, r in enumerate(results, 1):
        m = r["metrics"]
        total_quality += m.get("quality", 0)
        total_words += m.get("length", 0)
        total_latency += m.get("latency", 0)
        if m.get("sentences_ok", True) and m.get("no_ai_disclaimer", True):
            constraint_ok += 1

        eval_data["results"].append({
            "question": r["question"],
            "response": r["response"],
            "metrics": {
                "quality": m.get("quality", 0),
                "words": m.get("length", 0),
                "sentences": m.get("sentences", 0),
                "sentences_ok": m.get("sentences_ok", True),
                "latency": m.get("latency", 0),
                "no_ai_disclaimer": m.get("no_ai_disclaimer", True),
                "has_think_tags": m.get("has_think_tags", False),
            },
        })

        print(f"  [{i}/{len(questions)}] {r['question'][:60]}...")
        print(f"      Quality: {m.get('quality', 'N/A'):>6}  "
              f"Words: {m.get('length', 'N/A'):>4}  "
              f"Sentences: {m.get('sentences', 'N/A'):>2}  "
              f"Latency: {m.get('latency', 'N/A'):.1f}s")
        print()

    # Save standalone results
    with open(standalone_json_path, "w") as f:
        json.dump(eval_data, f, indent=2)
    print(f"Results saved to: {standalone_json_path}")

    # Generate a basic markdown report
    n = len(results)
    avg_quality = total_quality / n if n else 0
    avg_words = total_words / n if n else 0
    avg_latency = total_latency / n if n else 0

    report_lines = [
        f"# Standalone Evaluation Report: {NPC_KEY}",
        f"",
        f"- **Model:** {CANDIDATE_GGUF}",
        f"- **Questions evaluated:** {n}",
        f"- **Generated:** {eval_data['timestamp']}",
        f"",
        f"## Aggregate Metrics",
        f"",
        f"| Metric | Value |",
        f"| ------ | ----- |",
        f"| Avg Quality Score | {avg_quality:.1f} |",
        f"| Avg Word Count | {avg_words:.0f} |",
        f"| Avg Latency | {avg_latency:.1f}s |",
        f"| Constraints Passed | {constraint_ok}/{n} |",
        f"",
        f"## Per-Question Results",
        f"",
    ]

    for i, r in enumerate(results, 1):
        m = r["metrics"]
        report_lines.append(f"### {i}. {r['question']}")
        report_lines.append(f"")
        report_lines.append(f"**Response:** {r['response']}")
        report_lines.append(f"")
        report_lines.append(
            f"**Metrics:** quality={m.get('quality', 'N/A')} | "
            f"words={m.get('length', 'N/A')} | "
            f"sentences={m.get('sentences', 'N/A')} | "
            f"latency={m.get('latency', 'N/A')}s | "
            f"sentences_ok={m.get('sentences_ok', 'N/A')} | "
            f"no_ai={m.get('no_ai_disclaimer', 'N/A')}"
        )
        report_lines.append(f"")

    report_text = "\n".join(report_lines)
    with open(report_md, "w") as f:
        f.write(report_text)
    print(f"Report saved to: {report_md}")

print("\nEvaluation complete.")


In [ ]:
# @title 📊 Display Evaluation Results
# @markdown Print summary statistics, per-question breakdown, and feedback JSON structure.

import json
from pathlib import Path

output_dir = Path(f"eval/reports/{NPC_KEY}")
report_md = output_dir / "eval_report.md"
report_html = output_dir / "eval_report.html"
feedback_json_path = output_dir / "feedback.json"
standalone_json_path = output_dir / "standalone_results.json"

print("=" * 60)
print("EVALUATION RESULTS SUMMARY")
print("=" * 60)

# ────────────────────────────────────────
# Side-by-side comparison results (via feedback JSON)
# ────────────────────────────────────────
if BASELINE_GGUF and feedback_json_path.exists():
    with open(feedback_json_path) as f:
        feedback = json.load(f)

    total = feedback.get("total_examples", 0)
    bw = feedback.get("baseline_wins", 0)
    cw = feedback.get("candidate_wins", 0)
    ties = feedback.get("ties", 0)
    win_rate = feedback.get("win_rate", 0) * 100

    print(f"\n--- Overall Results ---")
    print(f"  Total examples:  {total}")
    print(f"  Baseline wins:   {bw}")
    print(f"  Candidate wins:  {cw}")
    print(f"  Ties:            {ties}")
    print(f"  Win rate:        {win_rate:.0f}%")

    # Weak concepts
    weak = feedback.get("weak_concepts", [])
    per_concept = feedback.get("per_concept", {})
    if weak:
        print(f"\n--- Weak Concepts ({len(weak)} found) ---")
        for wc in weak:
            info = per_concept.get(wc, {})
            print(f"  {wc}:")
            print(f"    win_rate:      {info.get('win_rate', 0):.0%}")
            print(f"    avg_quality:   {info.get('avg_candidate_quality', 0):.1f}")
            print(f"    violations:    {info.get('constraint_violations', 0)}")
    else:
        print(f"\n--- Concepts ---")
        print("  No weak concepts identified.")
        if per_concept:
            for concept, info in per_concept.items():
                print(f"  {concept}: win_rate={info.get('win_rate', 0):.0%}, "
                      f"avg_quality={info.get('avg_candidate_quality', 0):.1f}")

    # Print full markdown report summary section
    if report_md.exists():
        content = report_md.read_text()
        if "## Summary" in content:
            idx = content.index("## Summary")
            print(f"\n--- Markdown Report Summary ---")
            print(content[idx:])

# ────────────────────────────────────────
# Standalone results
# ────────────────────────────────────────
elif not BASELINE_GGUF and standalone_json_path.exists():
    with open(standalone_json_path) as f:
        data = json.load(f)

    results = data.get("results", [])
    n = len(results)
    print(f"\n--- Standalone Evaluation Summary ---")
    print(f"  Model:             {data.get('model', 'N/A')}")
    print(f"  Questions:         {n}")
    print(f"  Timestamp:         {data.get('timestamp', 'N/A')}")

    if n:
        total_quality = sum(r["metrics"]["quality"] for r in results)
        total_words = sum(r["metrics"]["words"] for r in results)
        total_latency = sum(r["metrics"]["latency"] for r in results)
        constraints_ok = sum(
            1 for r in results
            if r["metrics"].get("sentences_ok", True)
            and r["metrics"].get("no_ai_disclaimer", True)
        )

        print(f"\n  Aggregate Metrics:")
        print(f"    Avg Quality:        {total_quality / n:.1f}")
        print(f"    Avg Word Count:     {total_words / n:.0f}")
        print(f"    Avg Latency:        {total_latency / n:.1f}s")
        print(f"    Constraints Passed: {constraints_ok}/{n}")

        print(f"\n  Per-Question Breakdown:")
        print()
        for i, r in enumerate(results, 1):
            m = r["metrics"]
            print(f"  {i}. Q: {r['question'][:70]}")
            print(f"     Quality: {m.get('quality', 'N/A'):>6}  | "
                  f"Words: {m.get('words', 'N/A'):>4}  | "
                  f"Sent: {m.get('sentences', 'N/A'):>2}  | "
                  f"Lat: {m.get('latency', 'N/A'):.1f}s  | "
                  f"AI?: {not m.get('no_ai_disclaimer', True)}")
            print()

    # Show if report exists
    if report_md.exists():
        print(f"Full report: {report_md}")

# ────────────────────────────────────────
# HTML report link
# ────────────────────────────────────────
if report_html.exists():
    print(f"\nHTML report: {report_html}")
    size_kb = report_html.stat().st_size / 1024
    print(f"  ({size_kb:.1f} KB)")

# ────────────────────────────────────────
# No results found
# ────────────────────────────────────────
if not feedback_json_path.exists() and not standalone_json_path.exists():
    print("\nNo evaluation results found. Run the evaluation cell first.")

print("\nDone.")


In [ ]:
# @title ⬇️ Download Evaluation Reports
# @markdown Download the evaluation report (HTML/markdown) and feedback JSON to your local machine.

import os
import glob
from pathlib import Path

print("=" * 60)
print("DOWNLOAD EVALUATION ARTIFACTS")
print("=" * 60)

# Check runtime
is_colab = False
try:
    from google.colab import files  # type: ignore
    is_colab = True
except ImportError:
    pass

output_dir = f"eval/reports/{NPC_KEY}"

if not os.path.exists(output_dir):
    print(f"No reports directory found at: {output_dir}")
    print("Run the evaluation cell first.")
    raise SystemExit(0)

# Collect all downloadable artifacts
downloads = []

# HTML report
for f in sorted(glob.glob(os.path.join(output_dir, "*.html"))):
    downloads.append(f)

# Markdown report
for f in sorted(glob.glob(os.path.join(output_dir, "*.md"))):
    downloads.append(f)

# Feedback / standalone JSON
for json_file in ("feedback.json", "standalone_results.json"):
    jp = os.path.join(output_dir, json_file)
    if os.path.exists(jp):
        downloads.append(jp)

if not downloads:
    print(f"No downloadable artifacts found in {output_dir}.")
    raise SystemExit(0)

print(f"Found {len(downloads)} file(s) for download:\n")
for f in downloads:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.relpath(f)} ({size_kb:.1f} KB)")

if is_colab:
    print("\nDownloading files...")
    for f in downloads:
        files.download(f)
    print("\nAll downloads complete.")
else:
    print("\nNot running in Colab. Files are available locally at:")
    for f in downloads:
        print(f"  {f}")
